In [4]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

merged = pd.read_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/merged.parquet')

# one-hot encode location — LR can't use raw text categories
merged_encoded = pd.get_dummies(merged, columns=['location'], drop_first=True)

feature_cols = ['temperature', 'humidity', 'hour', 'day_of_week', 'month', 
                 'is_holiday', 'historical_avg_load'] + \
                [c for c in merged_encoded.columns if c.startswith('location_')]

X = merged_encoded[feature_cols]
y = merged_encoded['demand_gw']

In [5]:
# sort chronologically, then split — last ~15% as test set
merged_encoded = merged_encoded.sort_values('datetime')
split_idx = int(len(merged_encoded) * 0.85)

train_idx = merged_encoded.index[:split_idx]
test_idx = merged_encoded.index[split_idx:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"Train: {X_train['datetime'].min() if 'datetime' in X_train else ''}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train: 
Train size: 238312, Test size: 42056


In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Linear Regression — MAE: {mae:.2f} GW, RMSE: {rmse:.2f} GW")

Linear Regression — MAE: 10.48 GW, RMSE: 15.28 GW


In [8]:
X_test_with_meta = merged.loc[test_idx, ['location', 'datetime']].copy()
X_test_with_meta['actual'] = y_test.values
X_test_with_meta['predicted'] = preds
X_test_with_meta['abs_error'] = (X_test_with_meta['actual'] - X_test_with_meta['predicted']).abs()

per_location_mae = X_test_with_meta.groupby('location')['abs_error'].mean()
print(per_location_mae.sort_values())

X_test_with_meta['pct_error'] = (X_test_with_meta['abs_error'] / X_test_with_meta['actual']) * 100
print(X_test_with_meta.groupby('location')['pct_error'].mean().sort_values())

location
North-Eastern     2.325887
Eastern           3.423311
Northern          9.016806
Southern          9.078200
Western          10.217905
National         28.845016
Name: abs_error, dtype: float64
location
National          15.120766
Northern          15.534239
Eastern           16.556388
Western           16.892392
Southern          17.659853
North-Eastern    118.348572
Name: pct_error, dtype: float64
